# LASH Four-Dataset Harmonization Preprocessing v2

This notebook freezes a **single preprocessing contract** for Cluster 1, Cluster 2, BDG-Edu, and BDG-Dorm before any new test performance is inspected.

## Reviewer-facing fairness rules

- All four datasets end with the same base schema: `Date, Holi, Temp, Humi, WS, Consumption`.
- `Holi=1` includes **Saturday, Sunday, or a holiday**.
- `Temp` is °C, `Humi` is %, and `WS` is **m/s in all four datasets**.
- BDG wind speed is converted from km/h to m/s.
- BDG weather is aggregated by clock hour and missing hourly values are repaired by **time-based linear interpolation**.
- No median or mean fallback is used for missing weather.
- THI, WCT, calendar cycles, and historical-demand variables are reconstructed by one shared function.
- The revised main benchmark must use **historical-only weather**; observed target-period weather is not a primary input.

The common variable family follows the prior XELF / BiGTA-Net / PLOS ONE design:
`Hour_x, Hour_y, DOTW_x, DOTW_y, Holi, Temp, Humi, WS, THI, WCT, Cons_1, Holi_1, Cons_7, Holi_7, Cons_avg`.

Sources:
- https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0307654
- https://www.sciencedirect.com/science/article/pii/S2213138822009365
- https://www.mdpi.com/2079-8954/11/9/456
- https://www.sciencedirect.com/science/article/pii/S1876610217330047


## 1. Imports and robust file resolution


In [ ]:
from __future__ import annotations

import json, re, zipfile
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from IPython.display import display

SEARCH_ROOTS = [Path.cwd(), Path("/content"), Path("/mnt/data")]

def canonical_name(name: str) -> str:
    p = Path(name)
    stem = re.sub(r"\s*\(\d+\)$", "", p.stem).rstrip()
    return f"{stem}{p.suffix}".casefold()

def resolve_file(expected_name: str) -> Path:
    for root in SEARCH_ROOTS:
        exact = root / expected_name
        if exact.exists():
            return exact
    key = canonical_name(expected_name)
    candidates = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(
                p for p in root.iterdir()
                if p.is_file() and canonical_name(p.name) == key
            )
    candidates = sorted(set(candidates), key=lambda p: (len(p.name), p.name.casefold()))
    if not candidates:
        raise FileNotFoundError(expected_name)
    return candidates[0]

C1_PATH = resolve_file("Cluster 1.csv")
C2_PATH = resolve_file("Cluster 2.csv")
ARCHIVE_PATH = resolve_file("archive.zip")

OUTPUT_DIR = Path.cwd() / "outputs" / "LASH_4dataset_harmonized_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(C1_PATH)
print(C2_PATH)
print(ARCHIVE_PATH)


## 2. Harmonize the two Korean source datasets

Cluster 1 uses an interval-end timestamp convention, so its timestamp is shifted by -1 h.
The source holiday regime is retained after timestamp harmonization, and weekends are explicitly forced to `Holi=1`.

Cluster 2 already uses standard hourly components. Its source holiday flag is retained and weekends are also explicitly forced to `Holi=1`.

The Korean weather variables are complete in the supplied files; therefore **no interpolation or imputation is applied to them**.


In [ ]:
def harmonize_cluster1(path: Path):
    raw = pd.read_csv(path, encoding="utf-8-sig")
    ts = pd.to_datetime(raw["Date"], errors="raise") - pd.Timedelta(hours=1)

    frame = pd.DataFrame(index=pd.DatetimeIndex(ts, name="Date"))
    for col in ["Temp", "Humi", "WS", "Consumption"]:
        frame[col] = pd.to_numeric(raw[col], errors="raise").to_numpy()

    source_holi = pd.Series(
        pd.to_numeric(raw["Holi"], errors="raise").astype(int).to_numpy(),
        index=frame.index,
    )
    day_key = pd.Series(frame.index.normalize(), index=frame.index)
    daily_source_holi = source_holi.groupby(day_key).transform(
        lambda s: int(s.mode().iloc[0])
    ).astype(int)

    weekend = (frame.index.dayofweek >= 5).astype(int)
    frame["Holi"] = np.maximum(daily_source_holi.to_numpy(), weekend)

    return frame[["Holi","Temp","Humi","WS","Consumption"]].sort_index(), raw


def harmonize_cluster2(path: Path):
    raw = pd.read_csv(path, encoding="utf-8-sig")
    year = raw["Year"].astype(int)
    year = np.where(year < 100, year + 2000, year)

    ts = pd.to_datetime({
        "year": year,
        "month": raw["Month"].astype(int),
        "day": raw["Day"].astype(int),
        "hour": raw["Hour"].astype(int),
    }, errors="raise")

    frame = pd.DataFrame(index=pd.DatetimeIndex(ts, name="Date"))
    for col in ["Temp", "Humi", "WS", "Consumption"]:
        frame[col] = pd.to_numeric(raw[col], errors="raise").to_numpy()

    source_holi = pd.to_numeric(raw["Holi"], errors="raise").astype(int).to_numpy()
    weekend = (frame.index.dayofweek >= 5).astype(int)
    frame["Holi"] = np.maximum(source_holi, weekend)

    return frame[["Holi","Temp","Humi","WS","Consumption"]].sort_index(), raw

cluster1, cluster1_raw = harmonize_cluster1(C1_PATH)
cluster2, cluster2_raw = harmonize_cluster2(C2_PATH)

display(cluster1.head())
display(cluster2.head())


## 3. Select a coherent BDG university-oriented site without test-performance information

Selection uses only metadata and file availability. An eligible site must share weather, timezone, and schedule and must contain both academic and dormitory building uses.
The largest eligible group is selected.


In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    archive_names = set(zf.namelist())
    meta = pd.read_csv(zf.open("meta_open.csv"))

USE_CODES = ["UnivClass","UnivLab","Office","UnivDorm"]

university = meta.loc[
    meta["subindustry"].eq("College/University")
    & meta["primaryspaceuse_abbrev"].isin(USE_CODES)
].copy()

rows = []
for (wf, tz, sch), g in university.groupby(
    ["newweatherfilename","timezone","annualschedule"], dropna=False
):
    n_edu = int(g["primaryspaceuse_abbrev"].isin(["UnivClass","UnivLab","Office"]).sum())
    n_dorm = int(g["primaryspaceuse_abbrev"].eq("UnivDorm").sum())
    rows.append({
        "weather_file": wf,
        "timezone": tz,
        "schedule_file": sch,
        "n_eligible": int(len(g)),
        "n_edu": n_edu,
        "n_dorm": n_dorm,
        "weather_available": wf in archive_names,
        "schedule_available": sch in archive_names,
        "has_both_target_groups": n_edu > 0 and n_dorm > 0,
    })

site_candidates = pd.DataFrame(rows)
site_candidates["eligible"] = (
    site_candidates["weather_available"]
    & site_candidates["schedule_available"]
    & site_candidates["has_both_target_groups"]
)
site_candidates = site_candidates.sort_values(
    ["eligible","n_eligible","n_edu","n_dorm"],
    ascending=[False,False,False,False],
).reset_index(drop=True)

selected_site = site_candidates.loc[site_candidates["eligible"]].iloc[0]
display(site_candidates.head(15))
display(pd.DataFrame([selected_site]))


## 4. Aggregate BDG-Edu and BDG-Dorm

`BDG_Edu` is the hourly sum of university classrooms, laboratories, and offices.
`BDG_Dorm` is the hourly sum of university dormitories.

These are two aggregate load series derived from one coherent external university-oriented site.


In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    meta = pd.read_csv(zf.open("meta_open.csv"))

    selected_mask = (
        meta["subindustry"].eq("College/University")
        & meta["newweatherfilename"].eq(selected_site["weather_file"])
        & meta["timezone"].eq(selected_site["timezone"])
        & meta["annualschedule"].eq(selected_site["schedule_file"])
        & meta["primaryspaceuse_abbrev"].isin(USE_CODES)
    )
    selected_meta = meta.loc[selected_mask].copy()

    meter_series: Dict[str, pd.Series] = {}
    for uid in selected_meta["uid"]:
        raw = pd.read_csv(zf.open(f"{uid}.csv"))
        s = pd.Series(
            pd.to_numeric(raw[uid], errors="raise").to_numpy(dtype=float),
            index=pd.to_datetime(raw["timestamp"], errors="raise"),
            name=uid,
        ).sort_index()

        expected = pd.date_range(s.index.min(), s.index.max(), freq="h")
        assert s.index.equals(expected)
        assert not s.isna().any()
        meter_series[uid] = s

edu_uids = selected_meta.loc[
    selected_meta["primaryspaceuse_abbrev"].isin(["UnivClass","UnivLab","Office"]), "uid"
].tolist()
dorm_uids = selected_meta.loc[
    selected_meta["primaryspaceuse_abbrev"].eq("UnivDorm"), "uid"
].tolist()

bdg_edu_load = pd.concat([meter_series[u] for u in edu_uids], axis=1).sum(axis=1)
bdg_dorm_load = pd.concat([meter_series[u] for u in dorm_uids], axis=1).sum(axis=1)

display(selected_meta["primaryspaceuse_abbrev"].value_counts())


## 5. BDG weather: hourly clock-bin aggregation + time-based linear interpolation

BDG weather is irregularly sampled (commonly around xx:51 and sometimes more than once within the same hour).

Processing:
1. `-9999` and `-` are treated as missing.
2. `Calm` wind becomes 0.
3. Multiple observations in the same clock hour are averaged.
4. Internal hourly gaps are repaired by `interpolate(method="time")`.
5. No median/mean fallback is allowed.
6. BDG wind speed is converted from km/h to m/s.

This makes `Temp`, `Humi`, and `WS` physically and numerically comparable with the two Korean datasets.


In [ ]:
def clean_weather_numeric(series: pd.Series, calm_to_zero: bool = False) -> pd.Series:
    text = series.astype("string").str.strip()
    if calm_to_zero:
        text = text.replace({"Calm":"0","CALM":"0","calm":"0"})
    text = text.replace({"-":pd.NA, "":pd.NA})
    values = pd.to_numeric(text, errors="coerce")
    return values.mask(values.eq(-9999)).astype(float)

with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    raw_weather = pd.read_csv(zf.open(selected_site["weather_file"]))

weather = pd.DataFrame({
    "Temp": clean_weather_numeric(raw_weather["TemperatureC"]).to_numpy(),
    "Humi": clean_weather_numeric(raw_weather["Humidity"]).to_numpy(),
    "WS_kmh": clean_weather_numeric(raw_weather["Wind SpeedKm/h"], calm_to_zero=True).to_numpy(),
}, index=pd.to_datetime(raw_weather["timestamp"], errors="raise")).sort_index()

hourly_weather_raw = weather.resample("1h").mean()
missing_before = hourly_weather_raw.isna().sum()

hourly_weather = hourly_weather_raw.interpolate(
    method="time",
    limit_area="inside",
)

if hourly_weather.isna().any().any():
    raise RuntimeError(hourly_weather.isna().sum().to_dict())

hourly_weather["WS"] = hourly_weather.pop("WS_kmh") / 3.6

weather_audit = pd.DataFrame({
    "variable":["Temp","Humi","WS"],
    "missing_before_interpolation":[
        int(missing_before["Temp"]),
        int(missing_before["Humi"]),
        int(missing_before["WS_kmh"]),
    ],
    "missing_after_interpolation":[
        int(hourly_weather["Temp"].isna().sum()),
        int(hourly_weather["Humi"].isna().sum()),
        int(hourly_weather["WS"].isna().sum()),
    ],
})

display(weather_audit)
display(hourly_weather.head(10))


## 6. Common holiday semantics

For all four datasets:

`Holi = 1` when the day is Saturday, Sunday, or a holiday.

For BDG, the primary holiday definition uses weekends plus the observed U.S. federal holiday calendar.
The source academic Regular/Summer/Break/Holiday schedule is not used as the main Holi input because it mixes institutional regimes with public holidays.


In [ ]:
federal_holidays = USFederalHolidayCalendar().holidays(
    start=bdg_edu_load.index.min(),
    end=bdg_edu_load.index.max(),
).normalize()

bdg_holi = (
    (bdg_edu_load.index.dayofweek >= 5)
    | bdg_edu_load.index.normalize().isin(federal_holidays)
).astype(int)

print([d.strftime("%Y-%m-%d") for d in federal_holidays])


## 7. Assemble the four identical base schemas


In [ ]:
bdg_edu = pd.DataFrame({
    "Holi": bdg_holi,
    "Temp": hourly_weather["Temp"].reindex(bdg_edu_load.index).to_numpy(),
    "Humi": hourly_weather["Humi"].reindex(bdg_edu_load.index).to_numpy(),
    "WS": hourly_weather["WS"].reindex(bdg_edu_load.index).to_numpy(),
    "Consumption": bdg_edu_load.to_numpy(),
}, index=pd.DatetimeIndex(bdg_edu_load.index, name="Date"))

bdg_dorm = bdg_edu.copy()
bdg_dorm["Consumption"] = bdg_dorm_load.to_numpy()

datasets = {
    "Cluster_1_Harmonized": cluster1,
    "Cluster_2_Harmonized": cluster2,
    "BDG_Edu_Harmonized": bdg_edu,
    "BDG_Dorm_Harmonized": bdg_dorm,
}

for name, df in datasets.items():
    expected = pd.date_range(df.index.min(), df.index.max(), freq="h")
    assert df.index.equals(expected)
    assert not df.isna().any().any()
    assert ((df.index.dayofweek >= 5) <= (df["Holi"].to_numpy() == 1)).all()

display(pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "start": df.index.min(),
        "end": df.index.max(),
        "Holi_days": int(df["Holi"].resample("D").max().sum()),
        "WS_mean_mps": float(df["WS"].mean()),
        "Consumption_mean": float(df["Consumption"].mean()),
    }
    for name, df in datasets.items()
]))


## 8. Rebuild one common input-variable family for all four datasets

The same function constructs:
`Hour_x, Hour_y, DOTW_x, DOTW_y, Holi, Temp, Humi, WS, THI, WCT, Cons_1, Holi_1, Cons_7, Holi_7, Cons_avg`.

The deterministic calendar fields are rebuilt rather than trusting dataset-specific encodings.
Historical-demand variables are generated causally from each harmonized demand series.


In [ ]:
def add_common_features(base: pd.DataFrame) -> pd.DataFrame:
    x = base.copy()
    idx = x.index

    hour = idx.hour.to_numpy(dtype=float)
    dow = idx.dayofweek.to_numpy(dtype=float)

    x["Hour_x"] = np.sin(2*np.pi*hour/24.0)
    x["Hour_y"] = np.cos(2*np.pi*hour/24.0)
    x["DOTW_x"] = np.sin(2*np.pi*dow/7.0)
    x["DOTW_y"] = np.cos(2*np.pi*dow/7.0)

    T = x["Temp"].astype(float)
    H = x["Humi"].astype(float)
    V = x["WS"].astype(float).clip(lower=0)

    x["THI"] = (1.8*T + 32.0) - ((0.55 - 0.0055*H) * (1.8*T - 26.0))
    x["WCT"] = 13.12 + 0.6215*T - 11.37*np.power(V,0.16) + 0.3965*T*np.power(V,0.16)

    x["Cons_1"] = x["Consumption"].shift(24)
    x["Holi_1"] = x["Holi"].shift(24)
    x["Cons_7"] = x["Consumption"].shift(168)
    x["Holi_7"] = x["Holi"].shift(168)

    temp = x.assign(_Hour=idx.hour)
    x["Cons_avg"] = temp.groupby(
        ["_Hour","Holi"], sort=False
    )["Consumption"].transform(
        lambda s: s.shift(1).rolling(7, min_periods=7).mean()
    )

    return x[
        ["Hour_x","Hour_y","DOTW_x","DOTW_y","Holi",
         "Temp","Humi","WS","THI","WCT",
         "Cons_1","Holi_1","Cons_7","Holi_7","Cons_avg","Consumption"]
    ]

common_features = {name:add_common_features(df) for name,df in datasets.items()}
display(common_features["BDG_Dorm_Harmonized"].head(10))


## 9. THI/WCT unit and formula audit

The reconstructed THI/WCT values for Cluster 1 and Cluster 2 must reproduce the supplied source columns.
This is also a direct check that BDG wind speed has been converted to the correct common unit before WCT is derived.


In [ ]:
source_c1_idx = pd.to_datetime(cluster1_raw["Date"]) - pd.Timedelta(hours=1)
source_c1_thi = pd.Series(cluster1_raw["THI"].to_numpy(), index=source_c1_idx)
source_c1_wct = pd.Series(cluster1_raw["WCT"].to_numpy(), index=source_c1_idx)

year = cluster2_raw["Year"].astype(int)
year = np.where(year < 100, year + 2000, year)
source_c2_idx = pd.to_datetime({
    "year":year,
    "month":cluster2_raw["Month"].astype(int),
    "day":cluster2_raw["Day"].astype(int),
    "hour":cluster2_raw["Hour"].astype(int),
})
source_c2_thi = pd.Series(cluster2_raw["THI"].to_numpy(), index=source_c2_idx)
source_c2_wct = pd.Series(cluster2_raw["WCT"].to_numpy(), index=source_c2_idx)

formula_audit = pd.DataFrame([
    {
        "dataset":"Cluster 1",
        "max_abs_THI_error":float(np.max(np.abs(
            common_features["Cluster_1_Harmonized"]["THI"].to_numpy()
            - source_c1_thi.loc[cluster1.index].to_numpy()
        ))),
        "max_abs_WCT_error":float(np.max(np.abs(
            common_features["Cluster_1_Harmonized"]["WCT"].to_numpy()
            - source_c1_wct.loc[cluster1.index].to_numpy()
        ))),
    },
    {
        "dataset":"Cluster 2",
        "max_abs_THI_error":float(np.max(np.abs(
            common_features["Cluster_2_Harmonized"]["THI"].to_numpy()
            - source_c2_thi.loc[cluster2.index].to_numpy()
        ))),
        "max_abs_WCT_error":float(np.max(np.abs(
            common_features["Cluster_2_Harmonized"]["WCT"].to_numpy()
            - source_c2_wct.loc[cluster2.index].to_numpy()
        ))),
    }
])

display(formula_audit)
assert formula_audit[["max_abs_THI_error","max_abs_WCT_error"]].to_numpy().max() < 1e-6
print("PASS")


## 10. Export

The six-column files are the recommended inputs for the revised LASH benchmark.
The CommonFeatures files are exported for transparency only; the forecasting notebook should rebuild them with the shared function so that all four datasets are treated identically.


In [ ]:
for name, df in datasets.items():
    df.reset_index().to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

for name, df in common_features.items():
    df.reset_index().to_csv(OUTPUT_DIR / f"{name}_CommonFeatures.csv", index=False)

weather_audit.to_csv(OUTPUT_DIR / "BDG_weather_interpolation_audit.csv", index=False)
site_candidates.to_csv(OUTPUT_DIR / "BDG_site_selection_audit.csv", index=False)
formula_audit.to_csv(OUTPUT_DIR / "THI_WCT_unit_formula_audit.csv", index=False)

print("Saved to", OUTPUT_DIR)


## 11. Protocol to carry into the model rerun

1. Same six-column base schema for all four datasets.
2. Same feature-construction code for all four datasets.
3. `Holi=1` for weekends or holidays.
4. Temp in °C, Humi in %, WS in m/s.
5. Main benchmark uses historical-only weather.
6. No target-period observed weather in the headline experiment.
7. Same 168-hour lookback and direct 24-step formulation.
8. Same validation tuning / purged calibration logic.
9. No BDG test-informed preprocessing or site selection.
10. Freeze this preprocessing before model reruns.
